In [15]:
import torch
from torchvision import models, transforms
from PIL import Image
import cv2
import numpy as np
import urllib.request
import time
import os

In [16]:
# Ensure that you have the provided images in the same directory as the code
if not os.path.exists("aquarium.jpg"):
    raise FileNotFoundError(
        "\nPlease make sure you have the required input in this directory."
    )

if not os.path.exists("imagenet_classes.txt"):
    print("Downloading class labels...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
        "imagenet_classes.txt"
    )

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

In [17]:
#MODEL SETUP
print("Loading pre-trained ResNet18 model...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict(image_path):

    img = Image.open(image_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0)

    # TODO 4: Add timing logic here to measure inference latency (in milliseconds)
    start_time = time.perf_counter()  # Start timer
    with torch.no_grad():
        output = model(input_tensor)
    end_time = time.perf_counter()    # End timer

    latency_ms = (end_time - start_time) * 1000.0

    probs = torch.nn.functional.softmax(output[0], dim=0)
    confidence, predicted_idx = torch.max(probs, 0)

    return categories[predicted_idx.item()], confidence.item(), latency_ms

Loading pre-trained ResNet18 model...


In [18]:
def simulate_turbidity(img_array):
    """
    Simulate murky water (blur/haze).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # TODO 1: Implement blur or haze using cv2 or numpy
    blurred = cv2.GaussianBlur(img_array, (21, 21), 0)
    haze = np.full_like(img_array, (210, 200, 180), dtype=np.uint8)
    return cv2.addWeighted(blurred, 0.55, haze, 0.45, 0)

In [19]:
def simulate_color_shift(img_array):
    """
    Simulate depth color loss (attenuate the red channel).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # TODO 2: Implement red channel attenuation
    shifted = img_array.astype(np.float32).copy()
    shifted[:, :, 2] *= 0.15
    return np.clip(shifted, 0, 255).astype(np.uint8)

In [20]:
def simulate_sensor_noise(img_array):
    """
    Simulate low-light digital camera noise.
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # TODO 3: Add Gaussian noise to the image
    noise = np.random.normal(0, 35, img_array.shape)
    noisy = img_array.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

In [21]:
if __name__ == "__main__":

    all_images = [
        "aquarium.jpg",
        "jellyfish.jpg",
        "set_f20_SESR.png",
        "set_f46_SESR.png",
        "set_o20_SESR.png",
        "set_u106_SESR.png",
        "set_u113_SESR.png",
    ]

    for base_image in all_images:

        img = cv2.imread(base_image)
        stem = os.path.splitext(base_image)[0]

        cv2.imwrite(f"{stem}_turbid.jpg", simulate_turbidity(img.copy()))
        cv2.imwrite(f"{stem}_colorshift.jpg", simulate_color_shift(img.copy()))
        cv2.imwrite(f"{stem}_noise.jpg", simulate_sensor_noise(img.copy()))

        images_to_test = [
            ("Baseline (Clean)", base_image),
            ("Turbidity", f"{stem}_turbid.jpg"),
            ("Color Shift", f"{stem}_colorshift.jpg"),
            ("Sensor Noise", f"{stem}_noise.jpg")
        ]

        print(f"\n IMAGE: {base_image}")
        print("\nRUNNING VISION DIAGNOSTICS")

        for condition_name, file_path in images_to_test:
            label, conf, latency = predict(file_path)
            print(f"Condition : {condition_name}")
            print(f"Prediction: {label}")
            print(f"Confidence: {conf:.4f}")
            print(f"Latency   : {latency:.2f} ms")
            print("-" * 30)


 IMAGE: aquarium.jpg

RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: eel
Confidence: 0.4804
Latency   : 41.92 ms
------------------------------
Condition : Turbidity
Prediction: jacamar
Confidence: 0.0355
Latency   : 34.14 ms
------------------------------
Condition : Color Shift
Prediction: eel
Confidence: 0.5497
Latency   : 34.19 ms
------------------------------
Condition : Sensor Noise
Prediction: American chameleon
Confidence: 0.0863
Latency   : 38.15 ms
------------------------------

 IMAGE: jellyfish.jpg

RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: jellyfish
Confidence: 0.9949
Latency   : 34.31 ms
------------------------------
Condition : Turbidity
Prediction: rapeseed
Confidence: 0.0616
Latency   : 40.65 ms
------------------------------
Condition : Color Shift
Prediction: jellyfish
Confidence: 0.9939
Latency   : 38.50 ms
------------------------------
Condition : Sensor Noise
Prediction: jellyfish
Confidence: 0.5512
Latency   : 3